### 1st Test

In [ ]:
from dash import Dash, html
import dash_cytoscape as cyto

app = Dash(__name__)

#server = app.server


app.layout = html.Div([
    cyto.Cytoscape(
        id='cytoscape-two-nodes',
        layout={'name': 'preset'},
        style={'width': '100%', 'height': '400px'},
        elements=[
            {'data': {'id': 'one', 'label': 'Node 1'}, 'position': {'x': 75, 'y': 75}},
            {'data': {'id': 'two', 'label': 'Node 2'}, 'position': {'x': 200, 'y': 200}},
            {'data': {'source': 'one', 'target': 'two'}}
        ]
    )
])

if __name__ == '__main__':
    app.run(debug=True)

### 2nd Test

In [ ]:
from nodosFromTP import generate_TP_nodes_and_edges, strMateriasAprobadas, nodosDelPlan93 as nodosDelPlan


# from graphviz import Digraph
import numpy as np
import pandas as pd

# from datetime import datetime, timedelta

In [ ]:
year=[2018]
df_edges, dic_node = generate_TP_nodes_and_edges(year, None)
# df_edges, dic_node = generate_TP_nodes_and_edges(year, 3)

In [ ]:
# # Mati sugerencia: limitar a nodos con aristas de >= 3 personas
# df_edges = df_edges[df_edges['appears'] >= 3 ]

In [ ]:
def actualizar_elements_graph(yearInput):
    tpl, df_edges, dic_node = generate_TP_nodes_and_edges(yearInput, 3)

    cyto_nodes = [{"data": {"id": str(i), "label": str(i) +'\n' + strMateriasAprobadas(dic_node.loc[i]['vector']), 
                    "height": sum(dic_node.loc[i]['vector'])*15+60, #quantityMaterias height basado en #materias
                    "color": "#62b3ff" if(dic_node.loc[i]['vector'] in nodosDelPlan) else "#000000" }}
                    
                for i in tpl]
    
    # Translating to Cytoscape format:
    cyto_edges = [
        {"data": {"source": str(row.edge[0]), 
                "target": str(row.edge[1]), 
                'label_frequencyBased': row.appears, 
                'weight_frequencyBased': row.appears/2, 
                'color': "#62b3ff" if((dic_node.loc[row.edge[0]]['vector'] in nodosDelPlan) and 
                                        (dic_node.loc[row.edge[1]]['vector'] in nodosDelPlan) and 
                                        abs(nodosDelPlan.index(dic_node.loc[row.edge[0]]['vector']) - nodosDelPlan.index(dic_node.loc[row.edge[1]]['vector'])) == 1)  else "#5c5c5c",
                "idList": row.idList}} 
        for row in df_edges.itertuples() #if row.edge[0] not in idNodosDesconectados
    ]

    elements = cyto_nodes + cyto_edges
    return elements

#### Deprecated: ya se refactorizó en nodosFromTP.py. Comment old: se puede refactoring los siguientes 3 bloques que eliminan los nodos sueltos y las 2 lineas parche "tpl = [x for x in tpl if x not in idNodosDesconectados]" y " if row.edge[0] not in idNodosDesconectados"

In [ ]:
# ########### este anda 4/5/2026
def buscarNodosDesconectados(df_edges, dic_node, nodosDesconect):
    df_arrowIn = pd.DataFrame({
        'nodoID': dic_node.index.to_list(),
        'flechaIn': [False for _ in range(len(dic_node))]
        })

    for row in df_edges.itertuples():
        beginning = row.edge[0]
        end = row.edge[1]
        if beginning != end and beginning not in nodosDesconect: 
            df_arrowIn.loc[end,'flechaIn'] = True

    return df_arrowIn


df_flechaIn = buscarNodosDesconectados(df_edges, dic_node, [])

idNodosDesconectados = df_flechaIn[~df_flechaIn['flechaIn'] ].drop(0)['nodoID'].to_list()
oldLenNodosDesconectados =  -1

print(len(idNodosDesconectados))

while len(idNodosDesconectados) != oldLenNodosDesconectados:
    df_flechaIn_Again = buscarNodosDesconectados(df_edges, dic_node, idNodosDesconectados)

    oldLenNodosDesconectados = len(idNodosDesconectados)

    idNodosDesconectados = df_flechaIn_Again[~df_flechaIn_Again['flechaIn'] ].drop(0)['nodoID'].to_list()
    print(len(idNodosDesconectados))

In [ ]:
##init aca


# ########### este anda 22/5/2026
def buscarNodosDesconectados(df_edges, dic_node, nodosDesconect):
    indices = set(dic_node.index.to_list())
    indices.remove(0)

    tienenFlechaIn = set(df_edges[(df_edges['edge'].str[1] != df_edges['edge'].str[0]) & ~(df_edges['edge'].str[0].isin(nodosDesconect)) ]['edge'].str[1].to_list())

    return list(indices-tienenFlechaIn)


idNodosDesconectados = buscarNodosDesconectados(df_edges, dic_node, [])
oldLenNodosDesconectados =  -1

print(len(idNodosDesconectados))

while len(idNodosDesconectados) != oldLenNodosDesconectados:
    df_flechaIn_Again = buscarNodosDesconectados(df_edges, dic_node, idNodosDesconectados)

    oldLenNodosDesconectados = len(idNodosDesconectados)

    idNodosDesconectados = df_flechaIn_Again
    print(len(idNodosDesconectados))

In [ ]:
# tpl = set()
# for tuple in df_edges['edge'].tolist():
#     tpl.add(tuple[0])
#     tpl.add(tuple[1])
# #tpl son los index de los nodos que SI les llega o sale una edge de ellos                 # tpl = tpl - nodeExclude
# tpl = list(tpl)
# tpl = [x for x in tpl if x not in idNodosDesconectados]

_____________

In [ ]:
# df_flechaIn = pd.DataFrame({
#     'nodoID': dic_node.index.to_list(),
#     'flechaIn': [False for _ in range(len(dic_node))]
# })


# for row in df_edges.itertuples():
#     beginning = row.edge[0]
#     end = row.edge[1]
#     if beginning != end: 
#         df_flechaIn.loc[end,'flechaIn'] = True
    

In [ ]:
# idNodosDesconectados = df_flechaIn[~df_flechaIn['flechaIn'] ].drop(0)['nodoID'].to_list()
# len(idNodosDesconectados)

In [ ]:
# df_flechaIn = pd.DataFrame({
#     'nodoID': dic_node.index.to_list(),
#     'flechaIn': [False for _ in range(len(dic_node))]
# })


# for row in df_edges.itertuples():
#     beginning = row.edge[0]
#     end = row.edge[1]
#     if beginning != end and beginning not in idNodosDesconectados: 
#         df_flechaIn.loc[end,'flechaIn'] = True

# idNodosDesconectados = df_flechaIn[~df_flechaIn['flechaIn'] ].drop(0)['nodoID'].to_list()

# len(idNodosDesconectados)

In [ ]:
# dot = Digraph()
# for i in tpl: #range(len(dic_node)):
#     nombre =  strMateriasAprobadas(dic_node.loc[i]['vector'])

#     if(dic_node.loc[i]['vector'] in nodosDelPlan):
#         elColor="#62b3ff"
#     else:
#         elColor="#000000"
    
#     dot.node(str(i), str(i) +'\n' + nombre, color = elColor)

# for i in range(len(df_edges)):
#     beginning = df_edges['edge'].iloc[i][0]
#     end = df_edges['edge'].iloc[i][1]

#     width = df_edges['appears'].iloc[i]

#     if((dic_node.loc[beginning]['vector'] in nodosDelPlan) and 
#        (dic_node.loc[end]['vector'] in nodosDelPlan) and 
#        abs(nodosDelPlan.index(dic_node.loc[beginning]['vector']) - nodosDelPlan.index(dic_node.loc[end]['vector'])) == 1):
#         elColor="#62b3ff"
#     else:
#         elColor="#5c5c5c"

#     if(beginning in tpl and end in tpl):
#         dot.edge(str(beginning), str(end), penwidth = str( width/4 ), label=str( width ), color=elColor)


# dot

___________________
#### test examples 

In [ ]:
# # Example of what you have (or can generate):
# node_data = {
#     "0": "1. alg",
#     "1": "2. algn1",
#     "2": "3. anamat"
# }

# # Translating to Cytoscape format:
# cyto_nodes = [
#     {"data": {"id": node_id, "label": label, "quantityMaterias": 3}} 
#     for node_id, label in node_data.items()
# ]
# cyto_nodes

In [ ]:
# # What you have:
# raw_edges = [(0, 0), (0, 1), (1, 2), (2, 2)]

# # Translating to Cytoscape format:
# cyto_edges = [
#     {"data": {"source": str(src), "target": str(tgt), 'frequency': 1}} 
#     for src, tgt in raw_edges
# ]
# cyto_edges

In [ ]:

# for i in tpl: #range(len(dic_node)):
#     nombre =  strMateriasAprobadas(dic_node.loc[i]['vector'])

#     if(dic_node.loc[i]['vector'] in nodosDelPlan):
#         elColor="#62b3ff"
#     else:
#         elColor="#000000"
    
#     item= {"data": {"id": str(i), "label": str(i) +'\n' + nombre, 
#                     "quantityMaterias": sum(dic_node.loc[tpl[i]]['vector']), 
#                     "color": elColor}}
#     cyto_nodes.append(item)



______________
### Nodes

In [ ]:
# cyto_nodes = [{"data": {"id": str(i), "label": str(i) +'\n' + strMateriasAprobadas(dic_node.loc[i]['vector']), 
#                     "height": sum(dic_node.loc[i]['vector'])*15+60, #quantityMaterias height basado en #materias
#                     "color": "#62b3ff" if(dic_node.loc[i]['vector'] in nodosDelPlan) else "#000000" }}
                    
#                 for i in tpl]


In [ ]:
# # Translating to Cytoscape format:
# cyto_edges = [
#     {"data": {"source": str(row.edge[0]), 
#               "target": str(row.edge[1]), 
#               'label_frequencyBased': row.appears, 
#               'weight_frequencyBased': row.appears/2, 
#               'color': "#62b3ff" if((dic_node.loc[row.edge[0]]['vector'] in nodosDelPlan) and 
#                                     (dic_node.loc[row.edge[1]]['vector'] in nodosDelPlan) and 
#                                     abs(nodosDelPlan.index(dic_node.loc[row.edge[0]]['vector']) - nodosDelPlan.index(dic_node.loc[row.edge[1]]['vector'])) == 1)  else "#5c5c5c",
#               "idList": row.idList}} 
#     for row in df_edges.itertuples() #if row.edge[0] not in idNodosDesconectados
# ]
# cyto_edges[0]

## Ver mas tarde como usar HTML en PlotyDash https://www.youtube.com/watch?v=pLU7ZLPhyX8

###### ejm pavos:
- Latest: https://dash-example-index.herokuapp.com/bar-polar


- https://dash-example-index.herokuapp.com/knob-bar-chart
- https://dash-example-index.herokuapp.com/network-graphs
- https://dash-example-index.herokuapp.com/legend

- https://dash.plotly.com/basic-callbacks
- https://dash.plotly.com/interactive-graphing

- https://dash.plotly.com/cytoscape/reference/1000
"
selectedNodeData (list; optional): The list of data dictionaries of all selected nodes (e.g. using Shift+Click to select multiple nodes, or Shift+Drag to use box selection). Read-only.

selectedEdgeData (list; optional): The list of data dictionaries of all selected edges (e.g. using Shift+Click to select multiple nodes, or Shift+Drag to use box selection). Read-only."

In [ ]:

# 3. Dynamic Selection (Hover/Click)
# If you want a node to grow specifically when a user interacts with it, use the :selected or :active selectors in your stylesheet. This doesn't even require a Python callback!
# stylesheet = [
#     {
#         'selector': 'node',
#         'style': {
#             'height': '30px',
#             'width': '30px',
#             'transition-property': 'height, width',
#             'transition-duration': '0.3s'
#         }
#     },
#     {
#         'selector': 'node:selected',
#         'style': {
#             'height': '60px',
#             'width': '60px',
#             'background-color': 'red'
#         }
#     }
# ]

In [ ]:
import dash
from dash import html, dcc, Input, Output
import dash_cytoscape as cyto


app = dash.Dash(__name__, suppress_callback_exceptions=True)
server = app.server

# 1. Combine your translated nodes and edges
elements = actualizar_elements_graph(2016)

# 2. Define the layout
app.layout = html.Div(
    style={'fontFamily': 'Arial, sans-serif', 'padding': '10px'}, 
    children=[
    html.H1(
            "Grafo de Trayectorias Históricas Cs. Computación FCEyN-UBA", 
            style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '20px'}
        ),

    html.Div(
            style={
                'display': 'flex',          # This is the magic word for side-by-side!
                'flexDirection': 'row',     # Aligns children horizontally
                'gap': '20px',              # Adds space between the graph and the sidebar
                'height': '850px',           # Set a fixed height for the whole interactive area
                'marginTop': '20px',
                
                'position': 'relative', 
                'overflow': 'hidden'
            },
            children=[
            ### left
            html.Div(
                style={
                    'flex': '3',        # Grows to take up 3 parts of the available space
                    'border': '1px solid #e0e0e0',
                    'borderRadius': '8px',
                    'backgroundColor': '#f9f9f9'
                },
                children=[
                dcc.Loading(
                    id="loading-1",
                    type="circle", # Options: "graph", "cube", "circle", "dot", or "default"
                    delay_hide=190,
                    children= 
                    cyto.Cytoscape(
                        id='cytoscape-graph',
                        elements=elements,
                        style={'width': '100%', 'height': '1000px'},
                        
                        # 'breadthfirst' with directed=True mimics Graphviz's top-down look
                        layout={
                            'name': 'breadthfirst', 
                            'directed': True,
                            'roots': '[id = "0"]',
                            'spacingFactor': 0.85            
                        },
                        
                        # 3. Add basic styling to mimic your PDF
                        stylesheet=[
                            {
                                'selector': 'node',
                                'style': {
                                    'label': 'data(label)',
                                    'text-valign': 'top',
                                    'text-valign': 'center',
                                    'text-halign': 'center',
                                    'text-wrap': 'wrap',
                                    'text-max-width': '80px',

                                    'background-color': '#ffffff',
                                    'border-color': 'data(color)',
                                    'border-width': 2,
                                    'shape': 'rectangle',
                                    'width': 90,
                                    'height': 'data(height)',
                                }
                            },
                            {
                                'selector': 'edge',
                                'style': {
                                    'curve-style': 'bezier',
                                    'label': 'data(label_frequencyBased)',
                                    'width': 'data(weight_frequencyBased)',
                                    'target-arrow-shape': 'triangle',
                                    
                                    'line-color': 'data(color)',
                                    'target-arrow-color': 'data(color)',
                                    
                                    # 'loop-direction': '-45deg',
                                    'loop-sweep': '-45deg',
                                    'control-point-step-size': 90
                                }
                            },

                            # ==========================================
                            # 2. SELECTED STYLES
                            # ==========================================
                            
                            # # Selected Node Style
                            # {
                            # 'selector': 'node:selected',
                            # 'style': {
                            #     'background-color': '#3B82F6', # Bright blue background
                            #     'border-color': '#1D4ED8',     # Dark blue border
                            #     'border-width': 4,             # Thicker border
                            #     'color': '#FFFFFF',            # White text for contrast
                                
                            #     # Elevate the selected node above others
                            #     'z-index': 9999,
                                
                            #     # Optional: Add a slight shadow/glow effect (supported in newer Cytoscape.js)
                            #     'underlay-color': '#93C5FD',
                            #     'underlay-padding': 5,
                            #     'underlay-opacity': 0.5
                            # }
                            # },
                            
                            # Selected Edge Style
                            {
                                'selector': 'edge:selected',
                                'style': {
                                    'line-color': '#EF4444',       # Bright red line
                                    'target-arrow-color': '#EF4444', # Match arrow color to line
                                    'width': 'data(weight_frequencyBased)',
                                    
                                    # Elevate the selected edge above other edges
                                    'z-index': 9999
                                }
                            }
                        ]
                    )
                
                ),

                html.Div(
                            style={
                                'position': 'absolute',
                                'top': '20px',      # Floats 20px from the bottom
                                'left': '20px',        # Floats 20px from the left
                                'backgroundColor': 'rgba(255, 255, 255, 0.9)', # Slightly transparent white
                                'padding': '15px',
                                'border': '1px solid #d1d5db',
                                'borderRadius': '8px',
                                'boxShadow': '0 4px 6px rgba(0,0,0,0.1)',
                                'zIndex': 1000         # Ensures it sits ON TOP of the Cytoscape canvas
                            },
                            children=[                                
                                # Legend Item 1: Standard Node
                                html.Div(
                                    style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '8px'}, 
                                    children=[
                                        # The "Color Swatch" (Circle)
                                        html.Div(style={
                                            'width': '16px', 'height': '16px', #, 'borderRadius': '50%' 
                                            'backgroundColor': '#ffffff', 'border': '1px solid #000000', 'marginRight': '10px'
                                        }),
                                        html.Span("Materias Aprobadas", style={'fontSize': '14px', 'color': '#4b5563'})
                                    ]
                                ),

                                # # Legend Item 2: Selected Node
                                # html.Div(
                                #     style={'display': 'flex', 'alignItems': 'center', 'marginBottom': '8px'}, 
                                #     children=[
                                #         # The "Color Swatch" (Selected Circle)
                                #         html.Div(style={
                                #             'width': '16px', 'height': '16px', 'borderRadius': '50%', 
                                #             'backgroundColor': '#3B82F6', 'border': '2px solid #1D4ED8', 'marginRight': '10px'
                                #         }),
                                #         html.Span("Selected Subject", style={'fontSize': '14px', 'color': '#4b5563'})
                                #     ]
                                # ),

                                # Legend Item 3: Standard Edge
                                html.Div(
                                    style={'display': 'flex', 'alignItems': 'center'}, 
                                    children=[
                                        # The "Color Swatch" (Line)
                                        html.Div(style={
                                            'width': '20px', 'height': '7px', 'backgroundColor':"#62b3ff", 'marginRight': '6px'
                                        }),
                                        html.Span("Plan 93' sugerido", style={'fontSize': '14px', 'color': '#4b5563'})
                                    ]
                                )
                            ]
                        )

            ]),
            
            ### right
            html.Div(
                    style={
                        'flex': '1',        # Grows to take up 1 part of the available space
                        'padding': '20px',
                        'backgroundColor': '#ffffff',
                        'border': '1px solid #e0e0e0',
                        'borderRadius': '8px',
                        'boxShadow': '2px 2px 10px rgba(0,0,0,0.05)' # Gives it a nice pop
                    },
                    children=[
                        dcc.Dropdown(
                            id="dropdown-update-graph",
                            value=2016,
                            clearable=False,
                            searchable = False,
                            options=[
                                {"label": str(year), "value": year}
                                for year in range(2015, 2026, 1)
                            ],
                        ),

                    html.Pre(id='selected-nodes-output',
                     style= {'text-wrap': 'wrap'} )])

                
                
                
        ])
    
])

# ==========================================
# Seleccionar edge
# ==========================================
@app.callback(
    Output('selected-nodes-output', 'children'),
    Input('cytoscape-graph', 'selectedEdgeData') # List of selected edge data
)

def display_selected_edges(selected_edges):
    if not selected_edges:
        return f"Seleccione arista/s para más info\n",html.I("(se puede seleccionar más de una manteniendo Ctrl)")

    edge_ids = []

    for edge in selected_edges:
        edge_ids += edge.get('idList')

    uniqIDs = set(edge_ids)

    return "Cantidad de IDs Únicos: "+ str(len(uniqIDs))+ f"\n\n"+ f"IDs Estudiantes de Arista/s Seleccionadas: " + str(uniqIDs) 



# ==========================================
# Actualizar Graph por dropdown
# ==========================================
@app.callback(
    Output("cytoscape-graph", "elements"),
    # Output("cytoscape-graph", "layout"),
    Input("dropdown-update-graph", "value"),
)
def update_graph(value):
    # layout = {
    #     'name': 'breadthfirst',
    #     'animate': True,
    #     'animationDuration': 1000,

    #     'directed': True,
    #     'roots': '[id = "0"]',
    #     'spacingFactor': 0.85    
    # }

    elements = actualizar_elements_graph(value)
    return  elements #, layout

if __name__ == '__main__':
    app.run(debug=True)